In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('/home/mylinux/Desktop/MTech_Project/Self_Project/Vendor Invoice/data/inventory.db')
tables = pd.read_sql_query("select name from sqlite_master where type= 'table' ", conn)

In [3]:
tables

,name
0,purchases
1,purchase_prices
2,vendor_invoice
3,begin_inventory
4,end_inventory


In [4]:
for table in tables['name']:
    print('Table name is', table)
    df = pd.read_sql_query(f"select * from {table} limit 5", conn)
    display(df)

Table name is purchases


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


Table name is purchase_prices


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


Table name is vendor_invoice


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


Table name is begin_inventory


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01


Table name is end_inventory


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,2024-12-31
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,2024-12-31
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,2024-12-31
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,2024-12-31
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,2024-12-31


In [5]:

# 1. Load full vendor_invoice table
vendor_df = pd.read_sql_query("SELECT * FROM vendor_invoice;", conn)

# 2. Inspect shape and initial rows
print(f"Vendor Invoice Shape: {vendor_df.shape}")
display(vendor_df.head())

# 3. Check correlation between Quantity, Dollars, and Freight
corr_matrix = vendor_df[['Quantity', 'Freight', 'Dollars']].corr()
print("\nCorrelation Matrix:")
display(corr_matrix)

# 4. Feature Engineering: Freight per Unit & Bulk Discount Analysis
vendor_df['freight_per_unit'] = vendor_df['Freight'] / vendor_df['Quantity']

low_q_threshold = vendor_df['Quantity'].quantile(0.25)
high_q_threshold = vendor_df['Quantity'].quantile(0.75)

low_q_freight = vendor_df[vendor_df['Quantity'] <= low_q_threshold]['freight_per_unit'].mean()
high_q_freight = vendor_df[vendor_df['Quantity'] >= high_q_threshold]['freight_per_unit'].mean()

print(f"\nLow Quantity Threshold (25th percentile): {low_q_threshold}")
print(f"High Quantity Threshold (75th percentile): {high_q_threshold}")
print(f"Avg Freight per Unit (Low Quantity Vendors): ${low_q_freight:.4f}")
print(f"Avg Freight per Unit (High Quantity Vendors): ${high_q_freight:.4f}")


Vendor Invoice Shape: (5543, 10)


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN



Correlation Matrix:


,Quantity,Freight,Dollars
Quantity,1.000000,0.946550,0.963831
Freight,0.946550,1.000000,0.985141
Dollars,0.963831,0.985141,1.000000



Low Quantity Threshold (25th percentile): 83.0
High Quantity Threshold (75th percentile): 5100.5
Avg Freight per Unit (Low Quantity Vendors): $0.0946
Avg Freight per Unit (High Quantity Vendors): $0.0491


In [7]:
#Data Splitting and Model Training

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

In [8]:
vendor_df_clean = vendor_df.dropna(subset=['Quantity','Dollars','Freight']).copy()

X= vendor_df_clean[['Quantity','Dollars']]
y = vendor_df_clean['Freight']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8, random_state=42)
X_train.shape,X_test.shape

((4434, 2), (1109, 2))

In [10]:
models = { "Linear Regression": LinearRegression(), 
          "Decision Tree": DecisionTreeRegressor(random_state=42), 
          "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42) 
}

In [11]:
# Train and Evaluate Models
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })


In [12]:
results_df = pd.DataFrame(results)
display(results_df)

,Model,MAE ($),RMSE ($),R2 Score
0,Linear Regression,24.46,124.43,0.9700
1,Decision Tree,33.87,182.50,0.9355
2,Random Forest,27.65,138.64,0.9628


In [17]:

import joblib
from sklearn.model_selection import GridSearchCV

# 1. Hyperparameter Tuning for Random Forest
param_grid = {
    'n_estimators':[50,100,150],
    'max_depth': [10, 20, None],
    'min_samples_split':[2,5]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("Best Parameters for Random Forest:")
print(grid_search.best_params_)
print(f"Tuned Random Forest R2 Score: {r2_score(y_test, y_pred_best):.4f}")

# 2. Train and Save Final Best Model (Linear Regression)
final_model = LinearRegression()
final_model.fit(X_train, y_train)

# Save the trained model to disk
model_path = "/home/mylinux/Desktop/MTech_Project/Self_Project/Vendor Invoice/models/freight_model.pkl"
joblib.dump(final_model, model_path)
print(f"\nSuccessfully serialized final Freight Prediction model to: {model_path}")

# 3. Test Loading the Model & Running Sample Inference
loaded_model = joblib.load(model_path)
sample_data = pd.DataFrame({'Quantity':[1500], 'Dollars': [12000.00]})
predicted_freight = loaded_model.predict(sample_data)

print(f"\nSample Prediction Test:")
print(f"Inputs -> Quantity: 1,500 units | Order Value: $12,000.00")
print(f"Predicted Freight Cost: ${predicted_freight[0]:.2f}")

Best Parameters for Random Forest:
{'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Tuned Random Forest R2 Score: 0.9622

Successfully serialized final Freight Prediction model to: /home/mylinux/Desktop/MTech_Project/Self_Project/Vendor Invoice/models/freight_model.pkl

Sample Prediction Test:
Inputs -> Quantity: 1,500 units | Order Value: $12,000.00
Predicted Freight Cost: $65.18
